In [19]:
#Ignorar warnings
import warnings
warnings.filterwarnings("ignore")


In [20]:
import os

def find_project_root(marker_dirs=("artifacts", "data")):
    for root, dirs, files in os.walk("/", topdown=True):
        if all(m in dirs for m in marker_dirs):
            return root
    raise RuntimeError("No se encontró el project root con artifacts/ y data/")

print("Buscando project root... (puede tardar unos segundos)")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print("Project root detectado:", os.getcwd())
print("Contenido:", os.listdir("."))


Buscando project root... (puede tardar unos segundos)
Project root detectado: /mnt/batch/tasks/shared/LS_root/mounts/clusters/testingnewmodels/code/Users/jaragono/tfmg10-credit_scoring
Contenido: ['.amlignore', '.amlignore.amltmp', '.git', '.gitignore', '.gitignore.amltmp', '.ipynb_checkpoints', 'artifacts', 'configs', 'data', 'FETCH_HEAD', 'notebooks', 'src']


In [21]:
import joblib

xgb = joblib.load("artifacts/credit_model_xgb_20260113.pkl")
lgbm = joblib.load("artifacts/credit_model_lgbm_20260113.pkl")
gboost = joblib.load("artifacts/credit_model_gboost_20260113.pkl")

print("Modelos cargados correctamente")


Modelos cargados correctamente


In [36]:
def run_inference(model, X):
    pred = model.predict(X)
    proba = model.predict_proba(X)

    print("Pred:", int(pred[0]))
    print("Probabilidad de pago:", float(proba[0][1]) * 100, "%")


In [22]:
proba = xgb.predict_proba(X_test)
print(proba)

[[0.69407415 0.30592585]]


In [23]:
import pandas as pd

FEATURE_ORDER = [
    'edad',
    'ingresos_declarados',
    'nivel_endeudamiento',
    'utilizacion_tarjetas',
    'score_buro',
    'morosidad_prev',
    'consumo_electrico',
    'pago_serv_publico',
    'titularidad_serv_publico',
    'remesas',
    'ingresos_bancarios',
    'historial_empresa',
    'portabilidad',
    'ultimo_consumo_movil'
]

#Definir caso con mal crédito
test_case_raw = {
    'edad': 59,
    'ingresos_declarados': 350,
    'nivel_endeudamiento': 0.40,
    'utilizacion_tarjetas': 0.70,
    'score_buro': 513,
    'morosidad_prev': -3,
    'consumo_electrico': 53,
    'pago_serv_publico': 0,
    'titularidad_serv_publico': 0,
    'remesas': 0,
    'ingresos_bancarios': 400,
    'historial_empresa': 0,
    'portabilidad': 0,
    'ultimo_consumo_movil': 0
}

X_test = pd.DataFrame(
    [[test_case_raw[f] for f in FEATURE_ORDER]],
    columns=FEATURE_ORDER
)


In [34]:
# Reordenar exactamente como fue entrenado GradientBoosting
X_test_gboost = X_test[
    [
        'edad',
        'ingresos_declarados',
        'nivel_endeudamiento',
        'utilizacion_tarjetas',
        'score_buro',
        'morosidad_prev',
        'historial_empresa',
        'consumo_electrico',
        'pago_serv_publicos',
        'titularidad_serv_publico',
        'remesas',
        'ingresos_bancarios',
        'ultimo_consumo_movil',
        'portabilidad'
    ]
]

print(X_test_gboost.columns.tolist())


['edad', 'ingresos_declarados', 'nivel_endeudamiento', 'utilizacion_tarjetas', 'score_buro', 'morosidad_prev', 'historial_empresa', 'consumo_electrico', 'pago_serv_publicos', 'titularidad_serv_publico', 'remesas', 'ingresos_bancarios', 'ultimo_consumo_movil', 'portabilidad']


In [67]:
# Correr los 3 modelos con el mismo dataset
# Dataset con mal crédito

print("---------XGBoost---------")
run_inference(xgb, X_test)

print("---------LightGBM---------")
run_inference(lgbm, X_test_gboost)

print("---------GradientBoosting---------")
run_inference(gboost, X_test_gboost)


---------XGBoost---------
Pred: 0
Probabilidad de pago: 27.056846022605896 %
---------LightGBM---------
Pred: 0
Probabilidad de pago: 5.965412546579122 %
---------GradientBoosting---------
Pred: 0
Probabilidad de pago: 14.393397474712359 %


In [60]:
import pandas as pd

def build_X(test_case_raw, feature_order):
    return (
        pd.DataFrame([test_case_raw])
        .reindex(columns=feature_order, fill_value=0)
    )

FEATURE_ORDER = [
    'edad',
    'ingresos_declarados',
    'nivel_endeudamiento',
    'utilizacion_tarjetas',
    'score_buro',
    'morosidad_prev',
    'consumo_electrico',
    'pago_serv_publicos',
    'titularidad_serv_publico',
    'remesas',
    'ingresos_bancarios',
    'historial_empresa',
    'portabilidad',
    'ultimo_consumo_movil'
]


In [61]:
test_case_raw = {
    'edad': 35,
    'ingresos_declarados': 1550,
    'nivel_endeudamiento': 0.1711,
    'utilizacion_tarjetas': 0.20,
    'score_buro': 930,
    'morosidad_prev': 0,
    'consumo_electrico': 396,
    'pago_serv_publicos': 1,
    'titularidad_serv_publico': 1,
    'remesas': 500,
    'ingresos_bancarios': 2100,
    'historial_empresa': 1,
    'portabilidad': 1,
    'ultimo_consumo_movil': 80
}


In [62]:
X_test_good = build_X(test_case_raw, FEATURE_ORDER)

print(X_test_good.columns.tolist())

X_test_gboost_good = X_test_good[gboost.feature_names_in_]



['edad', 'ingresos_declarados', 'nivel_endeudamiento', 'utilizacion_tarjetas', 'score_buro', 'morosidad_prev', 'consumo_electrico', 'pago_serv_publicos', 'titularidad_serv_publico', 'remesas', 'ingresos_bancarios', 'historial_empresa', 'portabilidad', 'ultimo_consumo_movil']


In [68]:
# Correr los 3 modelos con el mismo dataset
# Dataset con mal crédito

print("---------XGBoost---------")
run_inference(xgb, X_test_good)

print("---------LightGBM---------")
run_inference(lgbm, X_test_gboost_good)

print("---------GradientBoosting---------")
run_inference(gboost, X_test_gboost_good)

---------XGBoost---------
Pred: 0
Probabilidad de pago: 27.056846022605896 %
---------LightGBM---------
Pred: 1
Probabilidad de pago: 60.107669506218755 %
---------GradientBoosting---------
Pred: 1
Probabilidad de pago: 84.58856347150446 %


In [70]:
test_case_raw = {
    'edad': 35,
    'ingresos_declarados': 1550,
    'nivel_endeudamiento': 0,
    'utilizacion_tarjetas': 0,
    'score_buro': 0,
    'morosidad_prev': 0,
    'consumo_electrico': 396,
    'pago_serv_publicos': 1,
    'titularidad_serv_publico': 1,
    'remesas': 0,
    'ingresos_bancarios': 2100,
    'historial_empresa': 1,
    'portabilidad': 1,
    'ultimo_consumo_movil': 80
}

In [71]:
#Entrenar sin credito
X_test_nc = build_X(test_case_raw, FEATURE_ORDER)

print(X_test_nc.columns.tolist())

X_test_gboost_nc = X_test_good[gboost.feature_names_in_]


['edad', 'ingresos_declarados', 'nivel_endeudamiento', 'utilizacion_tarjetas', 'score_buro', 'morosidad_prev', 'consumo_electrico', 'pago_serv_publicos', 'titularidad_serv_publico', 'remesas', 'ingresos_bancarios', 'historial_empresa', 'portabilidad', 'ultimo_consumo_movil']


In [72]:
# Correr los 3 modelos con el mismo dataset
# Dataset con mal crédito

print("---------XGBoost---------")
run_inference(xgb, X_test_nc)

print("---------LightGBM---------")
run_inference(lgbm, X_test_gboost_nc)

print("---------GradientBoosting---------")
run_inference(gboost, X_test_gboost_nc)

---------XGBoost---------
Pred: 0
Probabilidad de pago: 43.62518489360809 %
---------LightGBM---------
Pred: 1
Probabilidad de pago: 60.107669506218755 %
---------GradientBoosting---------
Pred: 1
Probabilidad de pago: 84.58856347150446 %
